In [1]:
import pandas as pd

In [2]:
output_path = "../f0_OSMOSYS_ECU_Output.csv"
#output_path = "../docs/PLANMICCOutput.csv"
current = pd.read_csv(output_path)
current

C:\Users\bj\AppData\Local\Temp\ipykernel_13228\814693311.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  current = pd.read_csv(output_path)


,Strategy,Future.ID,Fuel,Technology,Emission,Year,Demand,NewCapacity,AccumulatedNewCapacity,TotalCapacityAnnual,...,Capex2023,FixedOpex2023,VarOpex2023,Opex2023,Externalities2023,Capex_GDP,FixedOpex_GDP,VarOpex_GDP,Opex_GDP,Externalities_GDP
0,BAU,0,COLL_BLEND,BLEND_NO_DCOLL,NaN,2010,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DDP70,0,COLL_BLEND,BLEND_NO_DCOLL,NaN,2010,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DDP,0,COLL_BLEND,BLEND_NO_DCOLL,NaN,2010,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PA,0,COLL_BLEND,BLEND_NO_DCOLL,NaN,2010,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BAU,0,COLL_BLEND,BLEND_NO_DCOLL,NaN,2011,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27163,PA,0,NaN,NaN,NaN,2069,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27164,BAU,0,NaN,NaN,NaN,2070,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27165,DDP70,0,NaN,NaN,NaN,2070,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27166,DDP,0,NaN,NaN,NaN,2070,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
calibration_data_path = "calibration_data.csv"
calibration_df = pd.read_csv(calibration_data_path)
calibration_df

,Activity,SubCategory,Year,TotalEmission
0,5A,5A1,2010,892.43
1,5A,5A1,2012,1173.31
2,5A,5A1,2014,1286.31
3,5A,5A1,2016,2067.92
4,5A,5A1,2018,2433.74
5,5A,5A1,2020,2796.36
6,5A,5A1,2021,3028.88
7,5A,5A1,2022,3183.21
8,5A,5A2,2010,806.17
9,5A,5A2,2012,881.16


In [4]:
# Years
calibration_years = list(calibration_df['Year'].unique())

In [5]:
model_filters = (current['Year'].isin(calibration_years))&(current['Strategy'] == 'BAU') & (current['Emission'].str.startswith("CO2e")) & (~current['AnnualTechnologyEmission'].isna())
model_df = current.loc[model_filters, ['Year', 'Technology', 'Fuel', 'AnnualTechnologyEmission']]
model_df

,Year,Technology,Fuel,AnnualTechnologyEmission
9908,2010,COMPOST,NaN,0.001053
9916,2012,COMPOST,NaN,0.001410
9924,2014,COMPOST,NaN,0.002104
9932,2016,COMPOST,NaN,0.001892
9940,2018,COMPOST,NaN,0.002000
...,...,...,...,...
22312,2016,TWWWOT,NaN,0.610430
22320,2018,TWWWOT,NaN,0.470293
22328,2020,TWWWOT,NaN,0.485802
22332,2021,TWWWOT,NaN,0.493495


In [6]:
# Calibration 1A1
techs = {
    "5A1": ["LANDFILL"],
    "5A2": ["NO_CONTR_OD"],
    "5B1": ["COMPOST"],
    "5C1": ["INCIN"],
    "5D1": ["TWWWOT", "SEWERWW", "DWW_N"],
    "5D2": ["IWW"],
}

def check_calibration(sub_category:str):
    # Model Results
    model_filters = (model_df['Technology'].isin(techs[sub_category]))
    model_output = model_df.loc[model_filters, ['Year', 'Technology', 'AnnualTechnologyEmission']]
    model = model_output.groupby('Year')[['AnnualTechnologyEmission']].agg('sum').reset_index()
    model['AnnualTechnologyEmission'] = model["AnnualTechnologyEmission"] * 1000 # Mt to Gg
    model = model.rename(columns={"AnnualTechnologyEmission":"Model"})

    # check if all specified techs are found in the model
    model_techs = list(model_output['Technology'].unique())
    if len(model_techs) != len(techs[sub_category]):
        missing_techs = list(set(techs[sub_category]) - set(model_techs))
        print("WARNING: The following techs are missing in the model.")
        print(missing_techs)


    # Calibration Results
    calib = calibration_df.loc[calibration_df['SubCategory']==sub_category, ['Year', 'TotalEmission']]
    calib = calib.rename(columns={"TotalEmission":"INGEI"})

    result = pd.merge(model, calib, on='Year', how='inner')
    result['Error'] = (result['Model'] - result['INGEI'])/ result['INGEI'] * 100
    print(result)

In [7]:
print("*** 5.A.1. Sitios de disposición de residuos sólidos gestionados ***")
check_calibration("5A1")

*** 5.A.1. Sitios de disposición de residuos sólidos gestionados ***
   Year    Model    INGEI      Error
0  2010  1615.00   892.43  80.966574
1  2012  1807.87  1173.31  54.082894
2  2014  2477.42  1286.31  92.598985
3  2016  2538.32  2067.92  22.747495
4  2018  2772.35  2433.74  13.913154
5  2020  3092.41  2796.36  10.586977
6  2021  3010.21  3028.88  -0.616399
7  2022  3183.34  3183.21   0.004084


In [8]:
print("*** 5.A.2. Sitios de disposición de residuos sólidos no gestionados ***")
check_calibration("5A2")

*** 5.A.2. Sitios de disposición de residuos sólidos no gestionados ***
   Year    Model    INGEI         Error
0  2010  566.738   806.17 -2.969994e+01
1  2012  587.231   881.16 -3.335705e+01
2  2014  730.587  1160.81 -3.706231e+01
3  2016  600.323   692.77 -1.334454e+01
4  2018  617.983   500.64  2.343860e+01
5  2020  551.710   539.52  2.259416e+00
6  2021  513.928   468.00  9.813675e+00
7  2022  482.720   482.72 -1.177565e-14


In [9]:
print("*** 5.B.1 Compostaje ***")
check_calibration("5B1")

*** 5.B.1 Compostaje ***
   Year    Model  INGEI      Error
0  2010  1.05287   1.06  -0.672642
1  2012  1.40964   1.41  -0.025532
2  2014  2.10416   7.12 -70.447191
3  2016  1.89168   5.20 -63.621538
4  2018  1.99951   6.46 -69.047833
5  2020  1.98524   7.04 -71.800568
6  2021  1.89168   7.38 -74.367480
7  2022  2.62901   6.45 -59.240155


In [10]:
print("*** 5.C.1. Incineración de residuos ***")
check_calibration("5C1")

*** 5.C.1. Incineración de residuos ***
   Year    Model  INGEI      Error
0  2010  13.3661  13.37  -0.029170
1  2012  15.2755  14.44   5.786011
2  2014  16.5974  15.22   9.049934
3  2016  16.4506  15.12   8.800265
4  2018  17.0381  15.46  10.207633
5  2020  28.3478  21.82  29.916590
6  2021  13.3661  13.34   0.195652
7  2022  20.1226  17.20  16.991860


In [11]:
print("*** 5.D.1. Aguas residuales domésticas ***")
check_calibration("5D1")

*** 5.D.1. Aguas residuales domésticas ***
['DWW_N']
   Year     Model   INGEI      Error
0  2010  405.3344  405.26   0.018359
1  2012  533.4414  474.51  12.419422
2  2014  628.0394  487.37  28.862958
3  2016  672.8076  492.64  36.571858
4  2018  511.6715  505.78   1.164835
5  2020  513.2907  511.49   0.352050
6  2021  521.1481  518.66   0.479717
7  2022  564.5860  505.69  11.646661


In [12]:
print("*** 5.D.2. Aguas residuales industriales ***")
check_calibration("5D1")

*** 5.D.2. Aguas residuales industriales ***
['DWW_N']
   Year     Model   INGEI      Error
0  2010  405.3344  405.26   0.018359
1  2012  533.4414  474.51  12.419422
2  2014  628.0394  487.37  28.862958
3  2016  672.8076  492.64  36.571858
4  2018  511.6715  505.78   1.164835
5  2020  513.2907  511.49   0.352050
6  2021  521.1481  518.66   0.479717
7  2022  564.5860  505.69  11.646661
